# Ambiente conda limpo para treino (TensorFlow) e API (FastAPI)
Sequência mínima para criar um ambiente **o mais limpo possível** com Python 3.12 (sem pacotes padrão), ativá-lo e instalar **apenas** o necessário para treinar uma rede neural com TensorFlow, usar no **Jupyter Notebook** e expor a API com **FastAPI/Uvicorn**. Observação: mesmo com `--no-default-packages`, o conda ainda instala alguns pacotes de sistema (ex.: `openssl`, `ca-certificates`).

## 1. Criar ambiente (limpo) com Python 3.12
Cria o ambiente `aula_docker` sem bibliotecas padrão do conda.
```bash
conda create -n aula_docker python=3.12 --no-default-packages -y
```

## 2. Ativar o ambiente
Ativa o ambiente recém-criado para que os próximos comandos atuem nele.
```bash
conda activate aula_docker
```

## 3. Instalar o pip no ambiente
Adiciona o `pip` explicitamente ao ambiente limpo.
```bash
conda install pip -y
```

## 4. Atualizar o pip
Atualiza o `pip` para evitar problemas de compatibilidade.
```bash
python -m pip install --upgrade pip
```

## 5. Instalar ipykernel
Instala o kernel do Jupyter dentro do ambiente para uso nos notebooks.
```bash
pip install ipykernel
```

## 6. Instalar bibliotecas de treino
Instala dependências mínimas para treinar o modelo com TensorFlow + SciKeras + scikit-learn. (Se não houver wheel de TensorFlow para Python 3.12 no seu sistema, recrie o ambiente com `python=3.11`.)
```bash
pip install numpy pandas scikit-learn scikeras tensorflow joblib scipy cloudpickle
```

## 7. Instalar bibliotecas para a API e testes de requisição
Instala FastAPI, servidor ASGI (Uvicorn), validação (Pydantic), variáveis de ambiente e cliente HTTP.
```bash
pip install fastapi "uvicorn[standard]" pydantic python-dotenv requests
```


In [9]:
conda create -n aula_docker python=3.12 --no-default-packages -y
conda activate aula_docker
conda install pip -y
python -m pip install --upgrade pip
pip install ipykernel
pip install numpy pandas scikit-learn scikeras tensorflow joblib scipy cloudpickle
pip install fastapi "uvicorn[standard]" pydantic python-dotenv requests

SyntaxError: invalid syntax (2584511403.py, line 1)

# treinar_e_exportar_pipeline_regressao

## Objetivo
Treinar um **pipeline de regressão** para prever `age` usando uma **rede neural simples (TensorFlow)** integrada ao `sklearn`. O pipeline aplica **MinMaxScaler** nas colunas numéricas, **OneHotEncoder** nas categóricas e **normaliza o alvo** internamente com `TransformedTargetRegressor`. Ao final, avalia o desempenho (MAE, RMSE, R², MAPE), **exporta um único artefato** do pipeline para ser usado na API e retorna o caminho do arquivo e um DataFrame de métricas (treino e teste).

## Entradas (parâmetros da função)
- `caminho_csv`: caminho do arquivo CSV (ex.: `dados/fuma_e_bebe.csv`).
- `coluna_alvo`: nome da coluna alvo (padrão: `age`).
- `pasta_base` / `subpasta_modelo`: onde salvar o artefato (padrões: `artefatos` / `modelo` → `./artefatos/modelo/`).
- `test_size`: fração usada para teste (padrão: **0.4**).
- `random_state`: semente para reprodutibilidade.
- `epochs`: número de épocas do treino (padrão: **10**).
- `batch_size`: tamanho do lote (padrão: **512**).

## Saídas (retorno da função)
- **`caminho_pipeline`**: string com o caminho do arquivo do pipeline salvo.
- **`tabela_metricas`**: DataFrame com métricas em **`train`** e **`test`**:
  - **MAE** (erro absoluto médio), **RMSE** (raiz do erro quadrático médio), **R²**, **MAPE%**.
- Além disso, é gerado um `info_regressao.json` com colunas usadas e métricas.

## Fluxo detalhado (o que acontece por dentro)
1. **Carregar dados**: lê o CSV informado. A aula assume base **sem nulos** e sem inconsistências.
2. **Separar X e y**: `y = age` (float) e `X = demais colunas`.
3. **Detectar tipos de colunas**:
   - **Numéricas**: todas as colunas de tipo número.
   - **Categóricas**: colunas de texto/objeto.
4. **Pré-processamento mínimo** (aplicado automaticamente no treino e na predição):
   - **Numéricas**: **MinMaxScaler** para colocar cada coluna no intervalo `[0, 1]`.
   - **Categóricas**: **OneHotEncoder** com `handle_unknown="ignore"` e saída densa; isso evita erro se surgir uma categoria nova na predição e mantém o formato simples para estudo.
   - Não há imputação: a base da aula **não possui nulos** por premissa.
5. **Modelo (TensorFlow)**:
   - Rede neural **simples**: 1 entrada, duas camadas densas pequenas (`relu`) e uma saída linear (regressão).
   - Integração via **SciKeras (`KerasRegressor`)**, que permite encaixar o modelo no `Pipeline`.
   - **Verbose por época** ativado na chamada do `fit`, para exibir o progresso do treino.
6. **Normalização do alvo**:
   - Uso de `TransformedTargetRegressor` com **MinMaxScaler** em `y`. O alvo é normalizado só para treinar a rede e é **automaticamente revertido** nas predições.
7. **Montagem do `Pipeline`**:
   - Etapa 1: **pré-processamento de X** (OneHot + MinMax).
   - Etapa 2: **regressor** (rede neural com normalização do alvo).
   - Vantagem: o mesmo processamento usado no treino é aplicado na hora de prever, sem código extra.
8. **Divisão treino/teste**:
   - Separa 60% para treino e **40% para teste** (`test_size=0.4`).
9. **Treinamento**:
   - Chamada do `fit` com **validação interna** (`validation_split=0.1`) e **verbose=1** (mostra as épocas).
   - **EarlyStopping** com `patience=3` para parar cedo se não houver melhora.
10. **Avaliação**:
    - Faz predições em treino e teste.
    - Calcula **MAE**, **RMSE** (como raiz do MSE), **R²** e **MAPE%** (com proteção contra divisão por zero).
    - Organiza tudo em um **DataFrame** com duas linhas: `train` e `test`.
11. **Exportação do pipeline**:
    - O pipeline completo (pré-processamento + modelo + normalização do alvo) é salvo em `./artefatos/modelo/pipeline_referencia.pkl`.
    - A serialização usa **cloudpickle** para suportar o “builder” do Keras definido dentro da função.
12. **Salvar metadados (opcional)**:
    - `info_regressao.json` com nomes de colunas numéricas/categóricas e as métricas de `train` e `test`.
13. **Retorno**:
    - Caminho do arquivo salvo + DataFrame de métricas.

## Suposições e pré-requisitos
- Base **sem valores nulos** e esquema consistente entre treino e uso.
- Ambiente com **TensorFlow**, **SciKeras**, **scikit-learn** e **cloudpickle** instalados.
- Para **carregar na API**, use **cloudpickle** (e não `joblib`) para ler o arquivo salvo.

## Boas práticas (ideal no dia a dia)
- Criar e usar **ambiente virtual limpo** e registrar versões no `requirements.txt`.
- Executar **validação cruzada** e rodar mais épocas com **monitoramento** adequado quando houver tempo.
- Versionar artefatos em um **registry** (ex.: MLflow, DVC) e não depender de arquivos locais.
- Definir um **contrato de entrada** (schema) e validar dados antes de prever.

## Atalhos didáticos adotados aqui
- Rede neural **mínima** e poucas épocas para agilizar a aula.
- **`validation_split`** interno em vez de um conjunto de validação separado.
- **Serialização com cloudpickle** para simplificar a exportação mesmo com o builder do Keras dentro da função.

## O que não é recomendado em produção (e a alternativa)
- **Acoplar o `.pkl` diretamente ao repositório da API**:
  - Melhor: baixar o modelo de um **registry/armazenamento** controlado por versão.
- **Transportar pickles entre ambientes muito diferentes**:
  - Melhor: padronizar versões ou exportar o **modelo Keras** também em formato nativo (`model.save(...)`) e documentar o pré-processamento.


# Por que colocar `import` dentro das funções?

## Contexto
Em algumas partes do notebook, os `import` ficam **dentro das funções** (e não no topo do arquivo). Isso é uma escolha deliberada para este material didático.

## Benefícios (quando faz sentido)
- **Inicialização mais leve**: só carrega bibliotecas **quando a função é chamada**, reduzindo tempo de “start” do notebook/script.
- **Dependências opcionais**: permite que o arquivo rode mesmo se certas libs “pesadas” estiverem ausentes, desde que as funções que as usam **não sejam chamadas**.
- **Isolamento**: deixa claro **onde** cada dependência é usada (próximo da lógica), facilitando a leitura local daquela função.

## Malefícios (o que perde)
- **Padrão da comunidade**: a convenção mais comum é **importar no topo**; quem lê pode estranhar imports “espalhados”.
- **Ferramentas/IDE**: alguns linters, autocompletes e analisadores estáticos funcionam **melhor** com imports no topo.
- **Serialização**: certos objetos (ex.: *wrappers* de modelos) podem exigir cuidados extras ao salvar/carregar quando o código que os cria depende de funções locais ou imports condicionais.

## Boas práticas (recomendado)
- **Regra geral**: prefira **imports no topo** do arquivo para legibilidade e padronização.
- **Exceções úteis**:
  - Dependências **pesadas** e **opcionais** (ex.: frameworks de deep learning em passos específicos).
  - Caminhos de execução raros (códigos **não** utilizados na maioria das execuções).
- **Documente a escolha**: se usar import local, explique **por quê** (ex.: “carregar TensorFlow só ao treinar para reduzir tempo de carga do notebook”).
- **Evite em laços**: não faça import **dentro de loops**; mantenha-o no **início da função**.


## Aplicação neste material
- Usamos imports dentro das funções **para didática e leveza**: só carrega o necessário **quando** a etapa é executada (treino, avaliação, exportação).
- Registramos em Markdown o **motivo** e os **impactos**, para que os alunos saibam **quando** seguir (ou não) esse padrão no dia a dia.


In [10]:
def construir_tabela_metricas_regressao(y_treino, y_pred_treino, y_teste, y_pred_teste):
    """
    Constrói uma tabela (DataFrame) com métricas de regressão e estatísticas do alvo
    para TREINO e TESTE. Métricas com 2 casas decimais; 'quantidade' (n) inteiro;
    'mediana' inteira; 'média' e 'desvio_padrão' com 2 casas decimais.

    Retorno:
      DataFrame indexado por ['train', 'test'] com colunas:
      ['quantidade', 'MAE', 'RMSE', 'R2', 'MAPE_%', 'média', 'mediana', 'desvio_padrão'].
    """
    # Imports locais
    import numpy as np
    import pandas as pd
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    # Utilitários de formato
    _r2 = lambda v: float(np.round(v, 2))     # 2 casas decimais
    _i  = lambda v: int(v)                    # inteiro

    # Garante arrays 1D
    y_tr = np.asarray(y_treino).ravel()
    y_pr_tr = np.asarray(y_pred_treino).ravel()
    y_te = np.asarray(y_teste).ravel()
    y_pr_te = np.asarray(y_pred_teste).ravel()

    # Métricas com proteção no MAPE
    def _metricas(y_true, y_pred):
        mae = mean_absolute_error(y_true, y_pred)
        mse = mean_squared_error(y_true, y_pred)
        rmse = mse ** 0.5
        r2 = r2_score(y_true, y_pred)
        denom = np.clip(np.abs(y_true), 1e-8, None)
        mape = np.mean(np.abs((y_true - y_pred) / denom)) * 100.0
        return _r2(mae), _r2(rmse), _r2(r2), _r2(mape)

    mae_tr, rmse_tr, r2_tr, mape_tr = _metricas(y_tr, y_pr_tr)
    mae_te, rmse_te, r2_te, mape_te = _metricas(y_te, y_pr_te)

    # Estatísticas do alvo
    media_tr = _r2(np.mean(y_tr))
    media_te = _r2(np.mean(y_te))
    mediana_tr = _i(np.median(y_tr))
    mediana_te = _i(np.median(y_te))
    dp_tr = _r2(np.std(y_tr, ddof=1))
    dp_te = _r2(np.std(y_te, ddof=1))

    # Quantidades
    qtd_tr = _i(y_tr.shape[0])
    qtd_te = _i(y_te.shape[0])

    # Tabela final
    tabela = pd.DataFrame(
        {
            "quantidade": [qtd_tr, qtd_te],
            "MAE": [mae_tr, mae_te],
            "RMSE": [rmse_tr, rmse_te],
            "R2": [r2_tr, r2_te],
            "MAPE_%": [mape_tr, mape_te],
            "média": [media_tr, media_te],
            "mediana": [mediana_tr, mediana_te],
            "desvio_padrão": [dp_tr, dp_te],
        },
        index=["train", "test"]
    )
    return tabela


def treinar_e_exportar_pipeline_regressao(
    caminho_csv,
    coluna_alvo="age",
    pasta_base="artefatos",
    subpasta_modelo="modelo",
    test_size=0.5,
    random_state=42,
    epochs=10,
    batch_size=512
):
    """
    Regressão de idade com TensorFlow (rede neural simples) integrada a um Pipeline sklearn.
    Pré-processamento mínimo: One-Hot para categóricas e Min-Max para numéricas.
    Normalização do alvo dentro do pipeline via TransformedTargetRegressor.
    Retorna caminho do pipeline salvo (.pkl) e DataFrame com métricas (train/test).
    """
    # -------------------------------------------------------------------------
    # Imports locais para manter a função autocontida
    # -------------------------------------------------------------------------
    from pathlib import Path
    import json
    import numpy as np
    import pandas as pd

    from sklearn.model_selection import train_test_split
    from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    # SciKeras integra Keras ao ecossistema sklearn
    from scikeras.wrappers import KerasRegressor
    import tensorflow as tf
    import os
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
    from tensorflow.keras import layers, callbacks, optimizers

    # Para serializar pipeline com função local (evita PicklingError do joblib/pickle)
    import cloudpickle as cp  # requer: pip install cloudpickle

    # -------------------------------------------------------------------------
    # 1) Carregar dados — base sem nulos/inconsistências (premissa da aula)
    # -------------------------------------------------------------------------
    dados = pd.read_csv(caminho_csv)

    # -------------------------------------------------------------------------
    # 2) Separar alvo (y) e preditoras (X)
    # -------------------------------------------------------------------------
    y_alvo = dados[coluna_alvo].astype(float)   # idade contínua
    X_features = dados.drop(columns=[coluna_alvo])

    # -------------------------------------------------------------------------
    # 3) Identificar colunas por tipo
    # -------------------------------------------------------------------------
    colunas_numericas = X_features.select_dtypes(include=[np.number]).columns.tolist()
    colunas_categoricas = X_features.select_dtypes(exclude=[np.number]).columns.tolist()
#métrica vai pra baixo nesse caso, tem números que representam categoria e esse código não leva eles em consideração
#data: engenharia de atributos, contexto
    # -------------------------------------------------------------------------
    # 4) Pré-processamento mínimo (sem imputação; a base não tem nulos)
    #    - Numéricas: MinMaxScaler (0..1)
    #    - Categóricas: OneHotEncoder (ignora categorias novas)
    #    Comentários ao lado para clareza da turma.
    # -------------------------------------------------------------------------
    pipeline_numerico = Pipeline(steps=[
        ("escala_minmax", MinMaxScaler())                        # escala numéricas para [0, 1]
    ])
    pipeline_categorico = Pipeline(steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore",        # mapeia categorias em vetores binários
                                 sparse_output=False))           # saída densa (mais simples de entender)
    ])
    preprocessador_features = ColumnTransformer(
        transformers=[
            ("num", pipeline_numerico, colunas_numericas),       # aplica MinMax nas numéricas
            ("cat", pipeline_categorico, colunas_categoricas)    # aplica One-Hot nas categóricas
        ],
        remainder="drop"                                         # descarta qualquer coluna não listada
    )

    # -------------------------------------------------------------------------
    # 5) Definição do modelo TensorFlow (rede neural simples)
    #    Mantido DENTRO da função; SciKeras chamará este builder.
    # -------------------------------------------------------------------------
    def construir_modelo(meta):
        n_features = meta["n_features_in_"]  # número de colunas após o preprocessador
        modelo = tf.keras.Sequential([
            layers.Input(shape=(n_features,)),     # camada de entrada
            layers.Dense(32, activation="relu"),   # camada oculta pequena
            layers.Dense(16, activation="relu"),   # camada oculta ainda menor
            layers.Dense(1, activation="linear")   # saída contínua (idade)
        ])
        modelo.compile(
            optimizer=optimizers.Adam(learning_rate=1e-3),
            loss="mse",                 # MSE no espaço (normalizado pelo TTR)
            metrics=["mae"]             # MAE reportada durante o treino
        )
        return modelo

    regressao_tf = KerasRegressor(
        model=construir_modelo,         # builder local (mantido aqui por didática)
        epochs=epochs,
        batch_size=batch_size,
        verbose=0,                      # verbose controlado via parâmetros do fit
        random_state=random_state
    )

    # EarlyStopping para treino rápido e estável em aula
    parada_antecipada = callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    # -------------------------------------------------------------------------
    # 6) Normalização do alvo no Pipeline (TransformedTargetRegressor)
    #    Aplica MinMax em y durante treino e reverte automaticamente na predição.
    # -------------------------------------------------------------------------
    regressor_alvo_normalizado = TransformedTargetRegressor(
        regressor=regressao_tf,
        transformer=MinMaxScaler()
    )

    # Pipeline completo: pré-processa X e ajusta o regressor com y normalizado
    pipeline_modelo = Pipeline(steps=[
        ("preprocessar_features", preprocessador_features),  # One-Hot + MinMax em X
        ("regressor", regressor_alvo_normalizado)            # MLP(Keras) com y normalizado
    ])

    # -------------------------------------------------------------------------
    # 7) Divisão treino/teste (40% teste)
    # -------------------------------------------------------------------------
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_features, y_alvo, test_size=test_size, random_state=random_state
    )

    # -------------------------------------------------------------------------
    # 8) Treinamento com validação interna e verbose por época
    #    (usar regressor__validation_split/verbose/callbacks com SciKeras)
    # -------------------------------------------------------------------------
    pipeline_modelo.fit(
        X_tr, y_tr,
        regressor__validation_split=0.1,   # 10% do treino para validação
        regressor__verbose=1,              # mostra progresso por época
        regressor__callbacks=[parada_antecipada]
    )

    # -------------------------------------------------------------------------
    # 9) Avaliação — métricas em treino e teste (compatível com versões antigas)
    # -------------------------------------------------------------------------
    y_pred_tr = pipeline_modelo.predict(X_tr)
    y_pred_te = pipeline_modelo.predict(X_te)

    tabela_metricas = construir_tabela_metricas_regressao(y_tr, y_pred_tr, y_te, y_pred_te)

    # -------------------------------------------------------------------------
    # 10) Exportar pipeline — com cloudpickle (evita PicklingError do builder local)
    #     Observação: para carregar depois, use cloudpickle.load no serviço da API.
    # -------------------------------------------------------------------------
    pasta_saida = Path(f"{pasta_base}/{subpasta_modelo}")      # -> ./artefatos/modelo
    pasta_saida.mkdir(parents=True, exist_ok=True)
    caminho_pipeline = Path(f"{pasta_base}/{subpasta_modelo}/pipeline_referencia.pkl")
    with open(caminho_pipeline, "wb") as f:
        cp.dump(pipeline_modelo, f)  # serializa função local + pipeline completo

    # -------------------------------------------------------------------------
    # 11) Salvar JSON auxiliar (colunas e métricas)
    # -------------------------------------------------------------------------
    metricas_json = json.loads(tabela_metricas.to_json(orient="index"))

    info = {
        "alvo": coluna_alvo,
        "colunas_numericas": colunas_numericas,
        "colunas_categoricas": colunas_categoricas,
        "metricas": metricas_json
    }
    with open(pasta_saida / "info_regressao.json", "w", encoding="utf-8") as f:
        json.dump(info, f, ensure_ascii=False, indent=2)

    # -------------------------------------------------------------------------
    # 12) Retorno
    # -------------------------------------------------------------------------
    return str(caminho_pipeline), tabela_metricas


In [11]:
# Cria a pasta artefatos com a pipeline (modelo e processamento dos dados) e json com informações do treinamento
caminho_pipeline, df_metricas = treinar_e_exportar_pipeline_regressao(
    "dados/fuma_e_bebe.csv",
    coluna_alvo="age",
    test_size=0.5,
    epochs=50,
    batch_size=256
)

Epoch 1/50
349/349 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0955 - mae: 0.2196 - val_loss: 0.0281 - val_mae: 0.1343
Epoch 2/50
349/349 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0260 - mae: 0.1285 - val_loss: 0.0232 - val_mae: 0.1209
Epoch 3/50
349/349 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0225 - mae: 0.1190 - val_loss: 0.0218 - val_mae: 0.1173
Epoch 4/50
349/349 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0216 - mae: 0.1165 - val_loss: 0.0212 - val_mae: 0.1155
Epoch 5/50
349/349 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0211 - mae: 0.1152 - val_loss: 0.0208 - val_mae: 0.1142
Epoch 6/50
349/349 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0208 - mae: 0.1142 - val_loss: 0.0206 - val_mae: 0.1135
Epoch 7/50
349/349 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0206 - mae: 0.1135 - val_loss: 0.0204 - val_mae: 0.1129
Epoch 8/50
349/349 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0204 - mae: 0.1129 - val_loss: 0.0203 - val_mae: 0.1126
Epoch 9/50
349/349 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - lo

# Métricas de regressão — o que são e como interpretar

## Contexto deste pipeline
- **Problema**: regressão (prever um valor numérico contínuo, `age`).
- **Pré-processamento de X**: `MinMaxScaler` em numéricas; `OneHotEncoder` em categóricas.
- **Alvo (y)**: a função usa `TransformedTargetRegressor` para **normalizar o alvo apenas durante o treino** (escala 0–1) e **desnormalizar automaticamente** na hora de prever.  
  Resultado: as **métricas e previsões** ficam na **escala original da idade** (anos).

## Métricas reportadas
### 1) MAE — *Mean Absolute Error* (Erro Absoluto Médio)
- **O que mede**: média do módulo dos erros `|y - ŷ|`. Interpretação direta em **anos**.
- **Baixo é melhor**.  
- **Regra prática**: compare com a **escala natural** do alvo (p.ex., desvio-padrão de `age`) e com um **baseline ingênuo** (prever a média/mediana). Se o seu MAE for **bem menor** que o baseline, é bom.

### 2) RMSE — *Root Mean Squared Error* (Raiz do Erro Quadrático Médio)
- **O que mede**: raiz da média dos **erros ao quadrado**. Penaliza mais **erros grandes** (outliers).
- **Baixo é melhor**; mesma unidade do alvo (anos).  
- **Dica**: RMSE costuma ser **≥ MAE**. Se **RMSE ≫ MAE**, há indício de **outliers** ou alguns erros muito altos.

### 3) R² — *Coefficient of Determination* (Coeficiente de Determinação)
- **O que mede**: fração da variância de `y` explicada pelo modelo (entre -∞ e 1).
- **Alto é melhor**.  
- **Leituras usuais** (depende do domínio):  
  - `~0.0`: similar a prever a **média**.  
  - `0.3–0.5`: ganho modesto.  
  - `0.5–0.7`: **razoável/bom**.  
  - `>0.7`: **bom/alto**.  
  - **Negativo**: pior que prever a média.
  - Como se fosse uma acurácia da regressão

### 4) MAPE — *Mean Absolute Percentage Error* (Erro Percentual Absoluto Médio)
- **O que mede**: erro absoluto **em %** do valor real (usa proteção para evitar divisão por zero).
- **Baixo é melhor**.  
- **Regras gerais** (sensíveis ao domínio):  
  - `<10%`: **muito bom**  
  - `10–20%`: **ok/razoável**  
  - `20–50%`: **fraco**  
  - `>50%`: **ruim**

## Comparar treinamento × teste (generalização)
- **Treino ≈ Teste** (valores próximos): modelo **generaliza bem**; não há sobreajuste forte.
- **Treino melhor ≫ Teste**: provável **overfitting** (memoriza o treino).  
  Ações: mais regularização, **early stopping** (já aplicado), menos complexidade, mais dados.
- **Treino ruim e Teste ruim**: provável **underfitting** (modelo simples demais).  
  Ações: aumentar épocas/capacidade, melhorar features, revisar pré-processamento.

## O que é “bom” ou “ruim” aqui?
- **MAE/RMSE**: bons quando **baixos** **relativos ao problema** (anos). Compare com:
  1) **Baseline** (ex.: sempre prever a média da idade).  
  2) **Variação natural** da idade no dataset (desvio-padrão).
- **R²**: valores em torno de **0.5–0.7** indicam explicação **moderada a boa** da variação; `>0.7` é forte em muitos cenários.
- **MAPE**: `~10–20%` costuma ser **aceitável/razoável**; menor é melhor.

## Diagnóstico rápido usando as quatro métricas
- **RMSE muito maior que MAE** → erros grandes em alguns casos; ver outliers/segmentos.
- **R² baixo e MAE/RMSE altos** → revisar features, arquitetura/épocas ou qualidade dos dados.
- **MAPE alto com MAE razoável** → problema em faixas de valores pequenos (o % explode); avaliar segmentação.

## Conclusão prática
- Em regressão, **olhe as quatro métricas juntas**.  
- **MAE/RMSE** dizem “quantos **anos** erramos”, **R²** mostra “quanta **variância** explicamos” e **MAPE** traz a noção percentual.  
- **Treino vs Teste** parecidos indicam **bom equilíbrio**; discrepâncias guiam ajustes.


In [12]:
df_metricas

,quantidade,MAE,RMSE,R2,MAPE_%,média,mediana,desvio_padrão
train,99132,7.01,8.87,0.61,16.91,47.62,45,14.20
test,99132,7.06,8.94,0.60,17.09,47.52,45,14.13


# criar_api_ml_pronta

Gera uma **API pronta** para servir o pipeline de regressão. A função cria a pasta do projeto com os arquivos básicos, **copia o artefato do modelo** para dentro da API e produz um `requirements.txt` **enxuto** (somente dependências essenciais, nas mesmas versões do seu ambiente).

**Parâmetros**
- `nome_api`: nome da pasta/pacote da API (ex.: `api_idade`).
- `caminho_artefato_origem`: caminho do `.pkl` salvo no treino **ou** caminho de uma pasta que o contenha.  
  - Se for **arquivo**: copia direto para `artefatos/pipeline_referencia.pkl`.  
  - Se for **pasta**: copia a pasta, procura primeiro `pipeline_referencia.pkl` e, se não achar, o primeiro `.pkl` disponível.

**O que é criado**
- Estrutura mínima de pacote Python:
  - `main.py` → inicializa a API e define os endpoints.
  - `services/service.py` → carrega o pipeline e faz a predição.
  - `preprocessing/`, `models/`, `utils/` → pastas vazias de compatibilidade (mantêm a organização).
  - `artefatos/` → guarda o arquivo `.pkl` do pipeline.
- `requirements.txt` gerado via `pip freeze` e **filtrado por lista branca** (mantém só bibliotecas necessárias à API e ao pipeline; **não** altera versões).
- Cópia do pipeline para `artefatos/pipeline_referencia.pkl`.

**Como funciona (resumo direto)**
1. Cria a árvore de diretórios e arquivos básicos do pacote.
2. Copia o artefato informado (arquivo ou pasta) para `artefatos/` e garante `pipeline_referencia.pkl`.
3. Executa `pip freeze` no **ambiente atual** e filtra para ficar apenas o essencial (API + pipeline).
4. Escreve os arquivos-fonte da API com:
   - `GET /health` → retorna `{"status": "ok"}`.
   - `POST /predict` → recebe JSON com as **mesmas chaves do treino** (sem `age`) e retorna `{"idade_prevista": <valor>}`.
5. Deixa a API pronta para rodar **localmente**.

**Pré-requisitos**
- O `.pkl` do pipeline já foi gerado e está acessível no caminho informado.
- As dependências já foram instaladas no seu ambiente antes de rodar a função (o `freeze` lê deste ambiente).

**Como rodar local (somente na máquina)**
- Subir: `uvicorn <nome_api>.main:app --reload --host 127.0.0.1 --port 8010`  
- Testar: `http://127.0.0.1:8010/health` e `POST /predict`

**Estrutura gerada**
```text
📂 api_idade/
├── 📄 main.py
├── 📄 requirements.txt
├── 📄 __init__.py
│
├── 📂 artefatos/
│   └── 📄 pipeline_referencia.pkl
│
├── 📂 models/
│   └── 📄 __init__.py
│
├── 📂 preprocessing/
│   └── 📄 __init__.py
│
├── 📂 services/
│   ├── 📄 service.py
│   └── 📄 __init__.py
│
└── 📂 utils/
    └── 📄 __init__.py
```

**Sobre o `requirements.txt` limpo**
- Vem do seu ambiente (`pip freeze`) e é **filtrado** para manter só o que a API/pipeline usam (ex.: `fastapi`, `uvicorn`, `pydantic`, `requests`, `python-dotenv`, `numpy`, `pandas`, `scipy`, `scikit-learn`, `tensorflow`, `scikeras`, `cloudpickle`).  

**Limitações e riscos**
- Colocar o `.pkl` **dentro** da API é didático, **não** é prática de produção (o ideal é buscar o modelo de um **registry**/storage versionado (ex.: MLflow, storage versionado)).
- Reprodutibilidade entre máquinas pode exigir padronização de ambiente (lockfile/containers/CI).

In [18]:
def criar_api_ml_pronta(nome_api, caminho_artefato_origem):
    """
    Cria a API completa (estrutura de pastas + arquivos) para servir um pipeline de REGRESSÃO.
    Copia o artefato (treinado com cloudpickle) para <nome_api>/artefatos/pipeline_referencia.pkl,
    gera um requirements.txt via `pip freeze` e LIMPA mantendo apenas bibliotecas essenciais
    (sem alterar versões).

    Parâmetros:
      - nome_api: nome do pacote/dir da API (ex.: 'api_idade').
      - caminho_artefato_origem: caminho do .pkl salvo pela função de treino (ou de uma pasta que o contenha).

    Saída:
      - None (cria pastas/arquivos no disco).
    """
    # -------------------------------------------------------------------------
    # IMPORTS locais (mantém a função autocontida)
    # -------------------------------------------------------------------------
    import os
    import sys, subprocess, shutil
    from pathlib import Path
    from textwrap import dedent

    # -------------------------------------------------------------------------
    # 1) Preparação: diretórios-base
    # -------------------------------------------------------------------------
    base = Path.cwd()
    pkg = Path(nome_api)
    artefatos_dir = pkg / "artefatos"
    pkg.mkdir(parents=True, exist_ok=True)

    # -------------------------------------------------------------------------
    # 2) Estrutura de diretórios e arquivos vazios
    # -------------------------------------------------------------------------
    estrutura = {
        "": ["__init__.py", "main.py", "requirements.txt"],
        "models": ["__init__.py", "model_loader.py"],
        "preprocessing": ["__init__.py", "preprocessing.py"],
        "services": ["__init__.py", "service.py"],
        "utils": ["__init__.py", "utils.py"],
        "artefatos": [],
    }
    for pasta, arquivos in estrutura.items():
        d = pkg / pasta
        d.mkdir(parents=True, exist_ok=True)
        for arq in arquivos:
            (d / arq).touch(exist_ok=True)

    # -------------------------------------------------------------------------
    # 3) Copiar/Mover ARTEFATO para <pkg>/artefatos/pipeline_referencia.pkl
    #    - Aceita caminho para arquivo (.pkl) OU para diretório contendo o .pkl
    # -------------------------------------------------------------------------
    caminho_artefato_origem = Path(caminho_artefato_origem)
    artefatos_dir.mkdir(parents=True, exist_ok=True)
    destino_pipeline = artefatos_dir / "pipeline_referencia.pkl"

    if caminho_artefato_origem.is_file():
        # Caso 1: veio o .pkl diretamente
        shutil.copy2(caminho_artefato_origem, destino_pipeline)
    elif caminho_artefato_origem.is_dir():
        # Caso 2: veio uma pasta — copia a pasta inteira para referência
        destino_pasta_copiada = artefatos_dir / caminho_artefato_origem.name
        if destino_pasta_copiada.exists():
            shutil.rmtree(destino_pasta_copiada)
        shutil.copytree(caminho_artefato_origem, destino_pasta_copiada)

        # Procura o pipeline dentro da pasta copiada
        candidato = None
        alvo_preferido = list(destino_pasta_copiada.rglob("pipeline_referencia.pkl"))
        if alvo_preferido:
            candidato = alvo_preferido[0]
        else:
            pkl_qualquer = list(destino_pasta_copiada.rglob("*.pkl"))
            if pkl_qualquer:
                candidato = pkl_qualquer[0]

        if candidato is None:
            raise FileNotFoundError(
                f"Nenhum arquivo .pkl encontrado em {caminho_artefato_origem}"
            )
        shutil.copy2(candidato, destino_pipeline)
    else:
        raise FileNotFoundError(f"Caminho do artefato inválido: {caminho_artefato_origem}")

    # -------------------------------------------------------------------------
    # 4) Gerar requirements.txt via `pip freeze` e LIMPAR por lista branca
    #    - Mantém apenas dependências ESSENCIAIS para rodar a API + pipeline
    #    - NÃO altera versões (preserva exatamente o que está no ambiente ativo)
    # -------------------------------------------------------------------------
    dst_req = pkg / "requirements.txt"
    try:
        with dst_req.open("w", encoding="utf-8", newline="\n") as fh:
            subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=fh, check=True)
    except Exception:
        with dst_req.open("w", encoding="utf-8", newline="\n") as fh:
            subprocess.run(["pip", "freeze"], stdout=fh, check=True)

    manter_pkgs = {
        # runtime da API
        "fastapi", "uvicorn", "pydantic", "requests", "python-dotenv",
        # runtime do pipeline (sklearn + tensorflow + scikeras + serialização)
        "numpy", "pandas", "scipy", "scikit-learn", "tensorflow", "scikeras", "cloudpickle"
    }

    linhas = dst_req.read_text(encoding="utf-8").splitlines()
    filtradas = []
    for ln in linhas:
        ln_strip = ln.strip()
        if not ln_strip or ln_strip.startswith("#"):
            continue
        lower = ln_strip.lower()
        # extrai nome até '==', '[' (extras), '@' (VCS/URL) ou espaço
        sep_pos = min([p for p in [lower.find("=="), lower.find("["), lower.find("@"), lower.find(" ")] if p != -1] or [len(lower)])
        nome_pkg = lower[:sep_pos]
        if nome_pkg in manter_pkgs:
            filtradas.append(ln_strip)

    if not filtradas:
        # fallback mínimo para não deixar o arquivo vazio
        for ln in linhas:
            lw = ln.lower()
            if lw.startswith("fastapi") or lw.startswith("uvicorn"):
                filtradas.append(ln.strip())
        # garante dependências do pipeline
        for ln in linhas:
            lw = ln.lower()
            if lw.startswith("scikit-learn") or lw.startswith("tensorflow") or lw.startswith("cloudpickle"):
                filtradas.append(ln.strip())

    dst_req.write_text("\n".join(filtradas) + "\n", encoding="utf-8")

    # -------------------------------------------------------------------------
    # 5) Arquivos-fonte da API (código pronto)
    #    - main.py: endpoints e bootstrap
    #    - services/service.py: carregar pipeline (cloudpickle) + prever
    #    - demais módulos mantidos por compatibilidade
    # -------------------------------------------------------------------------
    main_py = dedent(f"""\
        # -*- coding: utf-8 -*-
        def criar_app():
            # Imports locais (carregados somente quando a API sobe)
            from fastapi import FastAPI, Body
            from {nome_api}.services.service import carregar_artefatos, prever_um

            # Caminho do pipeline dentro do pacote (execução a partir da raiz do projeto)
            CAMINHO_PIPELINE = "{nome_api}/artefatos/pipeline_referencia.pkl"

            app = FastAPI(title="API de Previsão de Idade")

            @app.on_event("startup")
            def iniciar():
                # Carrega pipeline único (pré-processamento + modelo + normalização do alvo)
                carregar_artefatos(caminho_pipeline=CAMINHO_PIPELINE)

            @app.get("/health")
            def verificar():
                return {{"status": "ok"}}

            @app.post("/predict")
            def prever(payload: dict = Body(..., description="JSON com as chaves de entrada")):
                # payload -> dict com as colunas de entrada (iguais às do treino)
                idade_prevista = prever_um(payload)
                return {{"idade_prevista": idade_prevista}}

            return app

        # Instância padrão para uvicorn:  uvicorn {nome_api}.main:app --reload --host 127.0.0.1 --port 8010
        app = criar_app()
    """)

    service_py = dedent("""\
        # -*- coding: utf-8 -*-
        _pipeline = None  # guardará o pipeline carregado (com pré-processamento + modelo)

        def carregar_artefatos(caminho_pipeline):
            \"\"\"Carrega o pipeline serializado com cloudpickle (.pkl).\"\"\"
            # Imports locais
            import os
            os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
            import cloudpickle as cp
            global _pipeline
            with open(caminho_pipeline, "rb") as f:
                _pipeline = cp.load(f)

        def prever_um(payload):
            \"\"\"Converte o payload em DataFrame e executa predict no pipeline carregado.\"\"\"
            # Imports locais
            import pandas as pd
            global _pipeline
            if _pipeline is None:
                raise RuntimeError("Pipeline não carregado. Verifique o evento de startup.")
            df = pd.DataFrame([payload])         # um único registro
            y_pred = _pipeline.predict(df)       # retorna array com 1 valor
            return float(y_pred[0])
    """)

    preprocessing_py = dedent("""\
        # -*- coding: utf-8 -*-
        # Módulo mantido por compatibilidade com a estrutura original.
        # O pré-processamento é executado dentro do pipeline treinado.
    """)

    model_loader_py = dedent("""\
        # -*- coding: utf-8 -*-
        # Módulo mantido por compatibilidade; não utilizado no fluxo atual.
        def carregar_modelo(caminho_modelo):
            import cloudpickle as cp
            with open(caminho_modelo, "rb") as f:
                return cp.load(f)
    """)

    utils_py = dedent("""\
        # -*- coding: utf-8 -*-
        def carregar_artefato(caminho):
            import cloudpickle as cp
            with open(caminho, "rb") as f:
                return cp.load(f)
    """)

    # Grava arquivos
    (pkg / "main.py").write_text(main_py, encoding="utf-8")
    (pkg / "services" / "service.py").write_text(service_py, encoding="utf-8")
    (pkg / "preprocessing" / "preprocessing.py").write_text(preprocessing_py, encoding="utf-8")
    (pkg / "models" / "model_loader.py").write_text(model_loader_py, encoding="utf-8")
    (pkg / "utils" / "utils.py").write_text(utils_py, encoding="utf-8")

    # Garante __init__.py com conteúdo mínimo
    for sub in ["", "services", "preprocessing", "utils", "models"]:
        init_path = pkg / sub / "__init__.py"
        if init_path.stat().st_size == 0:
            init_path.write_text("# pacote\n", encoding="utf-8")

In [14]:
# Criou toda a API
criar_api_ml_pronta(
    nome_api="api_idade",
    caminho_artefato_origem="artefatos/modelo/pipeline_referencia.pkl"
) 
# Execução local:
# uvicorn api_idade.main:app --reload --host 127.0.0.1 --port 8010
# http://127.0.0.1:8010/health

---

In [15]:
import pandas as pd
import numpy as np
import requests


def extrair_payload_y(tabela, indice, coluna_alvo="age"):
    """
    Retorna (payload_sem_alvo, y_real) para a linha indicada.
    """
    linha = tabela.iloc[indice]                          # pega a linha pelo índice
    y_real = float(linha[coluna_alvo])                   # valor real do alvo
    payload = linha.drop(labels=[coluna_alvo]).to_dict() # remove o alvo do payload
    payload = {k: (v.item() if hasattr(v, "item") else v) for k, v in payload.items()}  # tipagem nativa p/ JSON
    return payload, y_real


def prever_api(payload, y_real, base_url="http://127.0.0.1:8010"):
    """
    Envia payload à API (/predict) e mostra y_real e y_pred (sem casas decimais).
    """
    url = f"{base_url}/predict"                          # endpoint da API
    try:
        resp = requests.post(url, json=payload, timeout=30)  # faz o POST
        data = resp.json() if resp.ok else {}                # extrai JSON se OK
        y_pred = float(data.get("idade_prevista", np.nan))   # pega a predição
    except Exception:
        y_pred = np.nan                                      # evita erro na tela

    y_pred_int = int(round(y_pred)) if np.isfinite(y_pred) else None  # sem casas decimais
    print(f"y_real = {y_real} | y_pred = {y_pred_int}")                    # mostra os valores


In [16]:
CAMINHO_CSV = "dados/fuma_e_bebe.csv"
df = pd.read_csv(CAMINHO_CSV)

In [17]:
payload, y_real = extrair_payload_y(df, 5)
prever_api(payload, y_real)

y_real = 35.0 | y_pred = None
